### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

    1. Tracking agent behavior with logging, analytics, and debugging.
    2. Transforming prompts, tool selection, and output formatting.
    3. Adding retries, fallbacks and early termination logic.
    4. Applying rate limit, guardrails and PII detection.

In [3]:
from langchain_ollama import ChatOllama

# ollama model
ollama_model_agent = ChatOllama(model="qwen3:4b", temperature=0)
ollama_model_agent

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, model='qwen3:4b', temperature=0.0)

### Summarization MiddelWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following.

    1. Long-running conversations that execeed context windows.
    2. Multi-turn dialogues with extensive history
    3. Applications where preserving full conversation context matters

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent = create_agent(
    model=ollama_model_agent,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=ollama_model_agent,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [5]:
### Run with thread id
config = {"configurable": {"thread_id":"test-1"}}

In [ ]:
# Alternative test data
questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 100/4?",
    "what is 15-7?",
    "what is 3*3?",
    "what is 4*4"
]

for q in questions:
    response = agent.invoke( { "messages": [HumanMessage(content = q)]
        },
        config
    )
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='247d6e16-4d94-4ab1-a80d-5c36ed05bd03'), AIMessage(content="The answer to **2 + 2** is **4**.  \n\nThis is a fundamental result in basic arithmetic (addition). Here's a quick explanation:  \n- **2** represents two units.  \n- Adding **another 2** means combining two more units.  \n- Total = **2 + 2 = 4**.  \n\nThis holds true in standard mathematics, programming, and everyday contexts. 😊  \n\n*Hope this helps!*", additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-09-28T11:07:28.3788731Z', 'done': True, 'done_reason': 'stop', 'total_duration': 36213459200, 'load_duration': 12359000, 'prompt_eval_count': 17, 'prompt_eval_duration': 281907000, 'eval_count': 316, 'eval_duration': 35750659000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a0e7b2-397f-7df3-832f-1ebc12f1df78-0', tool_calls=[], invalid_tool_calls=[],

### Token Size

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 start, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, bussiness center
    3. Budget Stay - 3 star, $75/night, free"""

agent = create_agent(
    model = ollama_model_agent,
    tools = [search_hotel],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = ollama_model_agent,
            trigger = ("tokens",550),
            keep = ("tokens",200)
        )
    ]
)

### Run with thread id
config = {"configurable": {"thread_id":"test-1"}}

# Token counter(approximate)
def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token 

NameError: name 'ollama_model_agent' is not defined

In [1]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_token(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

NameError: name 'agent' is not defined

### Fraction

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 start, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, bussiness center
    3. Budget Stay - 3 star, $75/night, free"""

agent = create_agent(
    model = ollama_model_agent,
    tools = [search_hotel],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = ollama_model_agent,
            trigger = ("fraction", 0.005),# 0.5% = ~640 tokens
            keep = ("fraction", 0.002) # 0.2% = ~256 tokens
        )
    ]
)

### Run with thread id
config = {"configurable": {"thread_id":"test-1"}}

# Token counter(approximate)
def count_token(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token 

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_token(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_token(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-loop is useful for the following

    1. High-stakes operations requiring human approval (eg. database writes, financial transactions)
    2. Compliance workflows where human oversight is mandatory.
    3. Long-running conversations where human feedback guides the agent

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver 

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(reciptient: str, subject: str, body:str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {reciptient} with subject '{subject}'"

In [ ]:
agent = create_agent(
    model = ollama_model_agent,
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool":{
                    "allowed_decisions":["approved","edit","reject"]
                },
                "read_email_tool": False
            }
        )
    ]
)

In [ ]:
config = {"configurable": {"thread_id": "test-approved"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content = "Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [ ]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print(" pause! Approving...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ), 
        config = config
    )

    print(f"result: {result['messages'][-1].content}")

### Reject

In [ ]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content = "Send email to john@test.com with subject 'Hello' and body 'How are you?'")]}
)

In [ ]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print(" pause! Approving...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ), 
        config = config
    )

    print(f"result: {result['messages'][-1].content}")